# Magic / Dunder Methods

**OOP-in-Python — Intermediate**  
Customize object behavior with `__init__`, `__str__`, `__getitem__`, `__call__`, operator overloading, and more.

## What Are Dunder Methods?

**Dunder** = **D**ouble **Under**score — methods like `__init__`, `__str__`, `__add__`.

They allow your objects to behave like built-in types and integrate with Python syntax:

- `obj[key]` → `__getitem__`
- `len(obj)` → `__len__`
- `obj + other` → `__add__`
- `obj()` → `__call__`
- `with obj:` → `__enter__` / `__exit__`

## 1. Object Creation & Representation

| Method | Purpose | Called By |
|---|---|---|
| `__init__(self, ...)` | Constructor | `Obj()` |
| `__str__(self)` | Human-readable string | `str(obj)`, `print(obj)` |
| `__repr__(self)` | Unambiguous representation | `repr(obj)`, REPL |
| `__bool__(self)` | Truthiness | `bool(obj)`, `if obj:` |
| `__hash__(self)` | Hash for dict/set keys | `hash(obj)` |

In [ ]:
class Person:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    def __repr__(self) -> str:
        return f"Person('{self.name}', {self.age})"

    def __str__(self) -> str:
        return f"{self.name} ({self.age} years old)"

    def __bool__(self) -> bool:
        return self.age > 0  # Person is 'truthy' if age > 0

    def __hash__(self) -> int:
        return hash((self.name, self.age))

p = Person("Alice", 30)
print(repr(p))   # Person('Alice', 30)
print(str(p))    # Alice (30 years old)
print(bool(p))   # True
print(hash(p))   # Some integer hash

## 2. Container Emulation

| Method | Purpose | Called By |
|---|---|---|
| `__len__(self)` | Length | `len(obj)` |
| `__getitem__(self, key)` | Index/key access | `obj[key]` |
| `__setitem__(self, key, value)` | Index/key assignment | `obj[key] = val` |
| `__delitem__(self, key)` | Delete by key | `del obj[key]` |
| `__contains__(self, item)` | Membership | `item in obj` |
| `__iter__(self)` / `__next__` | Iteration | `for x in obj:` |

In [ ]:
class CustomList:
    def __init__(self, items: list = None):
        self._items = items or []

    def __len__(self) -> int:
        return len(self._items)

    def __getitem__(self, index: int):
        return self._items[index]

    def __setitem__(self, index: int, value):
        self._items[index] = value

    def __delitem__(self, index: int):
        del self._items[index]

    def __contains__(self, item) -> bool:
        return item in self._items

    def __iter__(self):
        return iter(self._items)

    def __repr__(self) -> str:
        return f"CustomList({self._items})"

cl = CustomList([10, 20, 30])
print(len(cl))         # 3
print(cl[1])           # 20
cl[1] = 25
print(25 in cl)        # True
for item in cl:
    print(item, end=' ')  # 10 25 30

## 3. Callable Objects (`__call__`)

Makes an instance callable like a function.

In [ ]:
class Multiplier:
    def __init__(self, factor: float):
        self.factor = factor

    def __call__(self, value: float) -> float:
        return value * self.factor

double = Multiplier(2)
triple = Multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15
print(callable(double))  # True

## 4. Context Managers (`__enter__` / `__exit__`)

Support the `with` statement for resource management.

In [ ]:
class FileManager:
    def __init__(self, filename: str, mode: str = 'r'):
        self.filename = filename
        self.mode = mode
        self.file = None

    def __enter__(self):
        self.file = open(self.filename, self.mode)
        return self.file  # Bound to 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
        # Return False to propagate exceptions, True to suppress
        return False

# Usage:
# with FileManager('test.txt', 'w') as f:
#     f.write('Hello')
# File is auto-closed after the block

## 5. Operator Overloading

| Method | Operator |
|---|---|
| `__add__(self, other)` | `+` |
| `__sub__(self, other)` | `-` |
| `__mul__(self, other)` | `*` |
| `__truediv__(self, other)` | `/` |
| `__eq__(self, other)` | `==` |
| `__lt__(self, other)` | `<` |
| `__le__(self, other)` | `<=` |
| `__gt__(self, other)` | `>` |
| `__ge__(self, other)` | `>=` |

In [ ]:
class Vector:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __add__(self, other: 'Vector') -> 'Vector':
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other: 'Vector') -> 'Vector':
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar: float) -> 'Vector':
        return Vector(self.x * scalar, self.y * scalar)

    def __eq__(self, other: 'Vector') -> bool:
        return self.x == other.x and self.y == other.y

    def __lt__(self, other: 'Vector') -> bool:
        # Compare by magnitude
        return (self.x**2 + self.y**2) < (other.x**2 + other.y**2)

    def __repr__(self) -> str:
        return f"Vector({self.x}, {self.y})"

v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2)   # Vector(4, 6)
print(v2 - v1)   # Vector(2, 2)
print(v1 * 3)    # Vector(3, 6)
print(v1 == Vector(1, 2))  # True
print(v1 < v2)   # True (magnitude: 5 < 25)

---
## Problem 1: Custom Vector Class (Full Operator Overloading)

**Difficulty: Medium**

Build a `Vector` class in 2D/3D with:
- Arithmetic: `+`, `-`, `*` (dot product and scalar), `/`
- Comparison: `==`, `!=`, `<`, `>` (by magnitude)
- `__abs__` for magnitude
- `__neg__` for negation
- `__getitem__` for index access

In [ ]:
import math

class Vector:
    def __init__(self, *components: float):
        self.comps = list(components)  # Support n-dimensional

    def __len__(self) -> int:
        return len(self.comps)

    def __getitem__(self, idx: int) -> float:
        return self.comps[idx]

    def __setitem__(self, idx: int, val: float):
        self.comps[idx] = val

    def __add__(self, other: 'Vector') -> 'Vector':
        return Vector(*[a + b for a, b in zip(self.comps, other.comps)])

    def __sub__(self, other: 'Vector') -> 'Vector':
        return Vector(*[a - b for a, b in zip(self.comps, other.comps)])

    def __mul__(self, other):
        # Scalar multiplication or dot product
        if isinstance(other, (int, float)):
            return Vector(*[c * other for c in self.comps])
        elif isinstance(other, Vector):
            return sum(a * b for a, b in zip(self.comps, other.comps))
        return NotImplemented

    def __rmul__(self, other):
        return self.__mul__(other)

    def __abs__(self) -> float:
        return math.sqrt(sum(c**2 for c in self.comps))

    def __neg__(self) -> 'Vector':
        return Vector(*[-c for c in self.comps])

    def __eq__(self, other: 'Vector') -> bool:
        return self.comps == other.comps

    def __lt__(self, other: 'Vector') -> bool:
        return abs(self) < abs(other)

    def __repr__(self) -> str:
        return f"Vector({', '.join(f'{c}' for c in self.comps)})"

v1 = Vector(1, 2, 3)
v2 = Vector(4, 5, 6)
print(v1 + v2)        # Vector(5, 7, 9)
print(v2 - v1)        # Vector(3, 3, 3)
print(v1 * 3)         # Vector(3, 6, 9)
print(v1 * v2)        # 32 (dot product: 1*4 + 2*5 + 3*6)
print(abs(v1))        # 3.741...
print(-v1)            # Vector(-1, -2, -3)
print(v1 < v2)        # True (smaller magnitude)
print(v1[0])          # 1 (via __getitem__)

## Problem 2: Custom Range Class

**Difficulty: Medium**

Implement a `Range` class that mimics Python's `range()` with iteration, indexing, `len()`, and `in`.

- **Time complexity**: `__getitem__` O(1), `__contains__` O(1), `__iter__` O(1) per step
- **Space complexity**: O(1) — lazy evaluation (no list storage)

In [ ]:
class Range:
    def __init__(self, start: int, stop: int = None, step: int = 1):
        if stop is None:
            start, stop = 0, start  # Range(n) becomes Range(0, n)
        if step == 0:
            raise ValueError("step cannot be 0")
        self.start = start
        self.stop = stop
        self.step = step

    def __len__(self) -> int:
        # Compute length without storing elements
        diff = self.stop - self.start
        if diff * self.step <= 0:
            return 0
        return (diff + self.step - (1 if self.step > 0 else -1)) // self.step

    def __getitem__(self, idx: int) -> int:
        if idx < 0:
            idx += len(self)
        if idx < 0 or idx >= len(self):
            raise IndexError("Range index out of range")
        return self.start + idx * self.step

    def __contains__(self, item: int) -> bool:
        if self.step > 0:
            return self.start <= item < self.stop and (item - self.start) % self.step == 0
        else:
            return self.stop < item <= self.start and (item - self.start) % self.step == 0

    def __iter__(self):
        current = self.start
        if self.step > 0:
            while current < self.stop:
                yield current
                current += self.step
        else:
            while current > self.stop:
                yield current
                current += self.step

    def __repr__(self) -> str:
        return f"Range({self.start}, {self.stop}, {self.step})"

r = Range(1, 10, 2)
print(len(r))           # 5
print(r[0], r[2])       # 1, 5
print(5 in r)           # True
print(4 in r)           # False
print(list(r))          # [1, 3, 5, 7, 9]
print(list(Range(5)))   # [0, 1, 2, 3, 4]
print(list(Range(10, 0, -2)))  # [10, 8, 6, 4, 2]

## Key Takeaways

- Dunder methods let your objects **integrate with Python syntax**.
- Container methods (`__getitem__`, `__len__`, `__iter__`) make objects behave like lists/dicts.
- Operator overloading (`__add__`, `__mul__`, etc.) enables intuitive APIs.
- `__call__` makes objects function-like — useful for factories and decorators.
- `__enter__`/`__exit__` enable context manager protocol (`with` statement).
- Lazy evaluation (like our `Range`) can save huge amounts of memory.